# Introduction to Deep Learning

This notebook covers fundamental deep learning concepts including:
- Why deep learning matters and when to use it
- Hierarchical representation learning
- Deep networks vs shallow networks
- Key concepts: activation functions, loss functions, backpropagation
- Transition from scikit-learn MLPClassifier to Keras

In [ ]:
# Import required libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras is available: {hasattr(tf, 'keras')}")

## What is Deep Learning?

**Deep Learning** is a subfield of machine learning using neural networks with multiple hidden layers (typically 3+). The "depth" enables learning hierarchical representations from raw data.

### Key Differences from Shallow Networks:

| Aspect | Shallow Networks | Deep Networks |
|--------|------------------|---------------|
| Hidden Layers | 1-2 layers | 3+ layers |
| Feature Learning | Hand-engineered features | Automatic feature discovery |
| Data Type | Structured/Tabular | Raw: images, text, audio |
| Parameter Count | Thousands | Millions to billions |
| Computation | CPU-friendly | GPU/TPU accelerated |
| Representation Learning | Limited | Hierarchical and rich |

### When to Use Deep Learning:
1. **Unstructured data**: images, text, audio, video
2. **Large datasets**: >100K samples
3. **Complex patterns**: need hierarchical features
4. **Transfer learning**: leveraging pre-trained models

## Hierarchical Representation Learning

One key power of deep learning is **representation learning** — the ability to automatically discover features needed for a task.

### Example: Handwritten Digit Recognition
```
Raw pixel data (784 values)
        ↓
Layer 1: learns edge detectors (horizontal, vertical, diagonal)
        ↓
Layer 2: learns stroke patterns (curves, loops, combinations)
        ↓
Layer 3: learns digit shapes (0-9 overall patterns)
        ↓
Output: digit class (0–9)
```

Each layer builds on the previous, creating increasingly abstract representations. This hierarchical approach is what separates deep learning from traditional ML where features are hand-engineered.

In [ ]:
# Visualize common activation functions
x = np.linspace(-5, 5, 100)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ReLU
axes[0].plot(x, np.maximum(0, x), 'b-', linewidth=2)
axes[0].set_title('ReLU: max(0, x)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))
axes[1].plot(x, sigmoid, 'g-', linewidth=2)
axes[1].set_title('Sigmoid: 1/(1+e^-x)', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')

# Tanh
tanh = np.tanh(x)
axes[2].plot(x, tanh, 'r-', linewidth=2)
axes[2].set_title('Tanh: (e^x - e^-x)/(e^x + e^-x)', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlabel('x')
axes[2].set_ylabel('f(x)')

plt.tight_layout()
plt.show()

print("Key Activation Functions:")
print("- ReLU (Rectified Linear): Most common for hidden layers, computes max(0, x)")
print("- Sigmoid: Used for binary classification output, squashes to [0, 1]")
print("- Softmax: Used for multi-class classification, probability distribution")
print("- Tanh: Similar to sigmoid but squashes to [-1, 1], often for hidden layers")

## Key Concepts in Training Deep Networks

### Loss Function
Measures how wrong the network's predictions are. Training minimizes this function.

**Common Loss Functions:**
- **Binary Crossentropy**: For binary classification (2 classes)
  - `L = -[y*log(ŷ) + (1-y)*log(1-ŷ)]`
- **Categorical Crossentropy**: For multi-class classification
  - `L = -Σ(y_i * log(ŷ_i))`  
- **Mean Squared Error (MSE)**: For regression
  - `L = (1/n)Σ(y - ŷ)²`

### Backpropagation
The algorithm for computing gradients of the loss with respect to every weight in the network, using the chain rule layer by layer from output back to input.

### Mini-batch SGD (Gradient Descent)
Weights are updated in the direction that reduces loss:
- Computed on small random subsets (batches) rather than full dataset
- Balances computational efficiency with gradient accuracy
- **Learning rate (η)**: Controls step size. Too high = oscillation, too low = slow training
- `w ← w − η × ∇L` (gradient descent update rule)

## From scikit-learn to Keras

### scikit-learn MLPClassifier (Shallow)
```python
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500)
mlp.fit(X_train, y_train)
score = mlp.score(X_test, y_test)
```

### Keras Sequential API (Deep)
```python
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)
```

**Key Differences:**
1. **Explicit layer construction** - Each layer is visible
2. **Activation functions** - Specified per layer
3. **Compilation step** - Choose optimizer and loss function
4. **Training details** - Control epochs, batch size, validation
5. **Flexibility** - Easy to add dropout, batch norm, custom layers

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset shape: {X.shape}")
print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# Train sklearn MLPClassifier
print("Training scikit-learn MLPClassifier...")
mlp_sklearn = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42, early_stopping=True)
mlp_sklearn.fit(X_train_scaled, y_train)
sklearn_train_score = mlp_sklearn.score(X_train_scaled, y_train)
sklearn_test_score = mlp_sklearn.score(X_test_scaled, y_test)

print(f"scikit-learn MLPClassifier:")
print(f"  Training accuracy: {sklearn_train_score:.4f}")
print(f"  Test accuracy: {sklearn_test_score:.4f}")
print(f"  Number of layers: {len(mlp_sklearn.coefs_)}")

In [ ]:
# Build equivalent model in Keras
print("\nBuilding Keras Sequential model...")
model_keras = tf.keras.Sequential([
    tf.keras.layers.Dense(100, activation='relu', input_shape=(30,)),
    tf.keras.layers.Dense(50, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile model
model_keras.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Print model summary
model_keras.summary()

In [ ]:
# Train Keras model
print("Training Keras model...")
history = model_keras.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluate on test set
keras_test_loss, keras_test_accuracy = model_keras.evaluate(X_test_scaled, y_test, verbose=0)
keras_train_loss, keras_train_accuracy = model_keras.evaluate(X_train_scaled, y_train, verbose=0)

print(f"\nKeras Sequential Model:")
print(f"  Training accuracy: {keras_train_accuracy:.4f}")
print(f"  Test accuracy: {keras_test_accuracy:.4f}")
print(f"  Test loss: {keras_test_loss:.4f}")

In [ ]:
# Compare performance
comparison_data = {
    'Model': ['scikit-learn', 'Keras'],
    'Train Accuracy': [sklearn_train_score, keras_train_accuracy],
    'Test Accuracy': [sklearn_test_score, keras_test_accuracy]
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Comparison bar plot
models = comparison_data['Model']
x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, comparison_data['Train Accuracy'], width, label='Train', alpha=0.8)
axes[0].bar(x + width/2, comparison_data['Test Accuracy'], width, label='Test', alpha=0.8)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Comparison: scikit-learn vs Keras')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim([0.95, 1.0])
axes[0].grid(True, axis='y', alpha=0.3)

# Training history
axes[1].plot(history.history['accuracy'], label='Train Accuracy', marker='o', markersize=3)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Keras Model Training History')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Both models achieve similar accuracy on the breast cancer dataset")
print("- Keras provides more control over training (epochs, batch size, validation)")
print("- Keras training history shows convergence behavior")
print("- The architecture is explicit in Keras, making it easier to modify")